# Atlas — слепой эксперимент Навье—Стокса

Этот notebook запускает **штатный** `AdaptiveResearchKernelOwner` из release `0.15.28.0` через модуль `evaluation.navier_stokes_blind_experiment`. Он содержит оба уровня: masked-term control и primitive-field blind operator discovery.

Atlas не получает название governing equation, формулу или расшифровку масок до окончания execution receipts. Проверяются две momentum-компоненты и локальное incompressibility closure на sealed holdout с невиданными параметрами reference world.

**Граница:** это benchmark восстановления структуры controlled reference world, а не доказательство существования/гладкости Навье—Стокса и не новый закон природы.


In [ ]:
from pathlib import Path
import json, sys

HERE = Path.cwd().resolve()
ROOT = next((p for p in (HERE, *HERE.parents) if (p / "source").is_dir() and (p / "evaluation").is_dir()), None)
if ROOT is None:
    raise RuntimeError("Atlas release root not found")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from source.lawspace.api import LawSpaceAPI
from evaluation.navier_stokes_blind_experiment import run, run_primitive_field

print("ROOT:", ROOT)
print("release:", LawSpaceAPI(ROOT).runtime.current_release_id())


## 1. Запуск полного blind benchmark

Следующая ячейка выполняет experiment одним вызовом и сохраняет полный JSON receipt. Не изменяйте `report` перед сохранением.


In [ ]:
report = run(ROOT)
print("status:", report["status"])
print("passed:", report["passed"], "/", report["total"])
print("digest:", report["digest"])


## 2. Ключевые результаты до физической интерпретации

Здесь показываются только masked coordinates и статистики Atlas.


In [ ]:
for lane in ("x_momentum", "y_momentum", "local_closure"):
    row = report["blind_results"][lane]
    print("\n", lane)
    print(" status:", row["status"])
    print(" activated:", row["activated_axes"])
    print(" initial NRMSE:", row["initial_holdout_nrmse"])
    print(" best:", row["best_expression"])
    print(" coeff:", row["best_coefficients_by_term"])
    print(" sealed NRMSE:", row["sealed_holdout"]["nrmse"])


## 3. Журнал исполнения

Журнал хранит последовательность freeze → Atlas execution → post-freeze decode и digest каждой стадии.


In [ ]:
for entry in report["run_journal"]:
    print(entry)


## 4. Post-freeze decode

Эта секция читается **после** того, как execution receipts уже сформированы. Она нужна только для проверки того, какую известную физическую структуру восстановил masked search.


In [ ]:
print(json.dumps(report["postfreeze_decoding"], ensure_ascii=False, indent=2))


## 5. Сохранение результата для передачи

Передайте созданный файл без ручного редактирования.


In [ ]:
OUT = ROOT / "reports" / "NAVIER_STOKES_BLIND_EXPERIMENT_CURRENT.json"
OUT.parent.mkdir(parents=True, exist_ok=True)
OUT.write_text(json.dumps(report, ensure_ascii=False, indent=2, sort_keys=True) + "\n", encoding="utf-8")
print("saved:", OUT)
print("bytes:", OUT.stat().st_size)


## Acceptance

Нормальный результат текущего control: `PASS_BLIND_CONTINUUM_BALANCE_RECOVERY`. Если статус другой, не подгоняйте параметры и не редактируйте данные. Сохраните JSON и terminal/notebook traceback — это и будет исследовательский результат.


# 6. Primitive-field blind discovery

Во втором уровне Atlas получает только обезличенные sampled fields и coordinate arrays. Производные, convective products, pressure-gradient coordinates и diffusion coordinates рождаются внутри Theory Compiler. `AXIS_BIRTH_CARDINALITY=ADAPTIVE`, поэтому один research cycle может активировать сразу несколько dormant axes.


In [ ]:
primitive_report = run_primitive_field(ROOT)
print('status:', primitive_report['status'])
print('passed:', primitive_report['passed'], '/', primitive_report['total'])
print('digest:', primitive_report['digest'])


## 7. Multi-axis birth и sealed OOD


In [ ]:
for lane in ('x_momentum', 'y_momentum'):
    row = primitive_report['blind_results'][lane]
    print('\n', lane)
    print(' born candidates:', row['born_candidate_axis_count'])
    print(' baseline:', row['baseline_axis'])
    print(' activated:', row['activated_axes'])
    print(' selected birth cardinality:', row['selected_birth_cardinality'])
    print(' discovery NRMSE:', row['discovery_holdout_nrmse'])
    print(' sealed NRMSE:', row['sealed_holdout']['nrmse'])
    print(' coefficients:', row['coefficients'])


## 8. Сохранение primitive-field receipt


In [ ]:
PRIMITIVE_OUT = ROOT / 'reports' / 'NAVIER_STOKES_PRIMITIVE_FIELD_CURRENT.json'
PRIMITIVE_OUT.parent.mkdir(parents=True, exist_ok=True)
PRIMITIVE_OUT.write_text(json.dumps(primitive_report, ensure_ascii=False, indent=2, sort_keys=True) + '\n', encoding='utf-8')
print(PRIMITIVE_OUT)


## Primitive-field acceptance

Нормальный статус текущего control: `PASS_PRIMITIVE_FIELD_BLIND_OPERATOR_DISCOVERY`. Если он не получен, не редактируйте fixture/JSON вручную: передайте полный receipt для разбора.
